# S004 — LSE PEAD / SUE — Drilldown
Set `TICKER` below, then **Run All**.

In [ ]:
TICKER = "HSBA"   # ← change and Run All

import sys, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

RUN_AT = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")

sys.path.insert(0, str(Path(r"c:\Personal\Business & Investments\Python codes")))
from signum import Chart
from signum.engine.dashboard import Dashboard
from signum.engine.statchart import StatChart

def _find_btest_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "AGENT_DSL_REFERENCE.md").exists(): return p
    return Path(r"c:\Personal\Business & Investments\Python codes\btest")

BTEST_ROOT  = _find_btest_root(); os.chdir(BTEST_ROOT)

SIGNAL_ROOT = Path("research/generated/Dividend Growth/signals/004_lse_pead")
OUTPUTS     = SIGNAL_ROOT / "outputs"
DATA_DIR    = SIGNAL_ROOT / "data"
SHARED_DATA = Path("research/generated/Dividend Growth/shared_data")

weights = pd.read_parquet(OUTPUTS / "weights.parquet")
trades  = pd.read_parquet(OUTPUTS / "trades.parquet")
equity  = pd.read_parquet(OUTPUTS / "equity.parquet")
sue     = pd.read_parquet(DATA_DIR / "sue_signal.parquet")

for df_ in [equity, weights, sue]:
    if hasattr(df_.index, "tz") and df_.index.tz is not None:
        df_.index = df_.index.tz_localize(None)

eq_col   = next((c for c in equity.columns if any(k in c.lower() for k in ("nav","portfolio","equity","total"))), equity.columns[0])
eq       = equity[eq_col].dropna()
prices_long = pd.read_parquet(SHARED_DATA / "lse_prices.parquet")

t_price  = prices_long[prices_long["ticker"]==TICKER].set_index("date")["close"].sort_index()
if hasattr(t_price.index, "tz") and t_price.index.tz is not None:
    t_price.index = t_price.index.tz_localize(None)

t_weight = weights[TICKER].fillna(0) if TICKER in weights.columns else pd.Series(dtype=float)
t_weight.index = pd.to_datetime(t_weight.index).normalize()
t_trades = trades[trades["instrument"]==TICKER].copy()
t_sue    = sue[TICKER].dropna() if TICKER in sue.columns else pd.Series(dtype=float)


Ticker: HSBA | Price rows: 3370 | Days held: 452 | Trades: 497 | SUE events: 903


---
## 1 · Price History + Trade Events

In [ ]:
price_df = pd.DataFrame({"time": t_price.index, "value": t_price.values})
hold_ser = t_weight.reindex(t_price.index, method="ffill").fillna(0)
hold_df  = pd.DataFrame({"time": hold_ser.index, "position": (hold_ser > 0.001).astype(int)})

chart = Chart(height=320, theme="dark",
              watermark=f"S004 · {TICKER} — Close Price GBX  ·  run {RUN_AT}")
chart.line(price_df, name="Close", color="#90caf9")
if hold_df["position"].any():
    chart.shade(hold_df, position_col="position")

if len(t_trades):
    buys  = pd.to_datetime(t_trades.loc[t_trades["side"].str.upper()=="BUY",  "datetime"]).dt.normalize()
    sells = pd.to_datetime(t_trades.loc[t_trades["side"].str.upper()=="SELL", "datetime"]).dt.normalize()
    if len(buys):
        chart.marker(pd.DataFrame({"time": buys.values,  "value": t_price.reindex(buys,  method="nearest").values}), shape="arrow_up",   color="#26a69a", text="Buy")
    if len(sells):
        chart.marker(pd.DataFrame({"time": sells.values, "value": t_price.reindex(sells, method="nearest").values}), shape="arrow_down", color="#ef5350", text="Sell")
chart


ValueError: Can only compare identically-labeled (both index and columns) DataFrame objects

---
## 2 · SUE (Standardised Unexpected Earnings) Over Time

In [ ]:
sue_df = pd.DataFrame({"time": t_sue.index, "value": t_sue.values}) if len(t_sue) else pd.DataFrame(columns=["time","value"])
wt_df  = pd.DataFrame({"time": t_weight.index, "value": t_weight.values * 100})

Dashboard(
    panes=[
        Chart(height=180).baseline(sue_df, base_value=0, value_col="value") if len(t_sue)
            else Chart(height=180).line(pd.DataFrame({"time":[], "value":[]}), name="no SUE data"),
        Chart(height=130).area(wt_df, name="Weight %", color="#1976d2"),
    ],
    titles=[
        f"S004 · {TICKER} — SUE at earnings  ·  {len(t_sue)} events  ·  run {RUN_AT}",
        f"{TICKER} — Portfolio Weight (%)",
    ],
    theme="dark",
)


---
## 3 · Earnings Events

In [ ]:
try:
    events   = pd.read_parquet(DATA_DIR / "events.parquet")
    t_events = events[events["ticker"]==TICKER].sort_values("report_date").copy()
    t_events["report_date"] = pd.to_datetime(t_events["report_date"])
    display(t_events.style
        .format({c: "{:.3f}" for c in t_events.select_dtypes("float").columns})
        .background_gradient(subset=["sue"] if "sue" in t_events.columns else [], cmap="RdYlGn", vmin=-3, vmax=3)
        .set_caption(f"S004 · {TICKER} — Earnings Events ({len(t_events)})")
        .hide(axis="index"))
except (FileNotFoundError, KeyError):
    pass


---
## 4 · Daily Return Contribution

In [ ]:
price_ret_tk  = t_price.pct_change(fill_method=None).clip(-0.5, 0.5)
t_wt_aligned  = t_weight.reindex(price_ret_tk.index, method="ffill").fillna(0)
daily_contrib = t_wt_aligned.shift(1).fillna(0) * price_ret_tk * 10000  # bps
cum_contrib   = daily_contrib.cumsum()

dc_df = pd.DataFrame({"time": daily_contrib.index, "value": daily_contrib.values})
cc_df = pd.DataFrame({"time": cum_contrib.index,   "value": cum_contrib.values})
total_bps = cum_contrib.dropna().iloc[-1] if len(cum_contrib.dropna()) else 0

Dashboard(
    panes=[
        Chart(height=160).histogram(dc_df, name="Daily contribution (bps)", color="#26a69a"),
        Chart(height=160).area(cc_df,      name="Cumulative (bps)",         color="#1976d2"),
    ],
    titles=[
        f"S004 · {TICKER} — Daily Contribution (bps)  ·  run {RUN_AT}",
        f"Cumulative  ·  total {total_bps:.0f} bps",
    ],
    theme="dark",
)


---
## 5 · Trade Log

In [ ]:
if len(t_trades):
    tlog = t_trades.copy()
    tlog["datetime"] = pd.to_datetime(tlog["datetime"]).dt.normalize()
    display(tlog.sort_values("datetime").style
        .format({c: "{:.3f}" for c in tlog.select_dtypes("float").columns})
        .set_caption(f"S004 · {TICKER} — All Trades ({len(tlog)})")
        .hide(axis="index"))
